# Posting — Run Locally in Jupyter

This notebook walks through the full pipeline step-by-step so you can inspect
research results, tweak topics/hooks, and preview slides interactively.

## 1. Install dependencies & set API keys

In [ ]:
# Install the project in editable mode (run once)
%pip install -e . jupyter

In [ ]:
import os

# Option A: load from .env file (recommended — copy .env.example to .env first)
try:
    %pip install python-dotenv -q
    from dotenv import load_dotenv
    load_dotenv()  # reads .env in the repo root
    print("Loaded .env file")
except ImportError:
    pass

# Option B: set keys directly (uncomment and fill in)
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["REDDIT_CLIENT_ID"]  = "..."
# os.environ["REDDIT_CLIENT_SECRET"] = "..."

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY before continuing"

## 2. Load config

In [ ]:
import yaml

with open("config.yaml") as f:
    config = yaml.safe_load(f)

slides_cfg   = config.get("slides", {})
research_cfg = config.get("research", {})
content_cfg  = config.get("content", {})
output_cfg   = config.get("output", {})

slide_count       = slides_cfg.get("count", 5)
tone              = slides_cfg.get("tone", "bold")
audience          = slides_cfg.get("audience", "retail investors")
colors            = slides_cfg.get("colors", {})
aspect_ratio      = slides_cfg.get("aspect_ratio", "9:16")
sources           = research_cfg.get("sources", ["news"])
topics            = research_cfg.get("topics", ["stocks"])
subreddits        = research_cfg.get("subreddits", ["stocks"])
review_iterations = content_cfg.get("review_iterations", 2)
style_notes       = content_cfg.get("style_notes", "")
output_dir        = output_cfg.get("directory", "./output")

print(f"Slide count: {slide_count}  |  Tone: {tone}  |  Aspect: {aspect_ratio}")
print(f"Sources: {sources}  |  Topics: {topics}")

## 3. Research — fetch trending finance topics

In [ ]:
from src.research.news import fetch_news_topics, format_news_for_prompt
from src.research.reddit import fetch_reddit_topics, format_reddit_for_prompt

research_parts = []

if "news" in sources:
    news_items = fetch_news_topics(topics)
    print(f"Found {len(news_items)} news articles")
    research_parts.append(format_news_for_prompt(news_items))

if "reddit" in sources:
    reddit_posts = fetch_reddit_topics(subreddits)
    print(f"Found {len(reddit_posts)} Reddit posts")
    research_parts.append(format_reddit_for_prompt(reddit_posts))

research_text = "\n\n".join(research_parts)
print(f"\nResearch text length: {len(research_text)} chars")

In [ ]:
# Preview the raw research (first 2000 chars)
print(research_text[:2000])

## 4. Extract structured facts

In [ ]:
from src.content.generator import extract_news_facts

try:
    research_facts = extract_news_facts(research_text)
    print(f"Extracted {len(research_facts)} facts")
    for fact in research_facts[:8]:
        print(f"  [{fact.get('type')}] {fact.get('fact')} ({fact.get('source')})")
except Exception as exc:
    print(f"Facts extraction failed: {exc}")
    research_facts = []

## 5. Suggest topics & pick one

In [ ]:
from src.content.generator import suggest_topics

topic_options = suggest_topics(research_text, audience)
for i, t in enumerate(topic_options, 1):
    print(f"{i}. {t['title']}: {t['description']}")

In [ ]:
# ✏️  Change the index to pick a different topic
TOPIC_INDEX = 0

chosen_topic = topic_options[TOPIC_INDEX]
angle = "Provide actionable insights with real data points"
print(f"Chosen topic: {chosen_topic['title']}")

## 6. Generate hooks & pick one

In [ ]:
from src.content.generator import generate_hooks

hook_options = generate_hooks(
    topic=chosen_topic["title"],
    angle=angle,
    tone=tone,
    audience=audience,
)
for i, h in enumerate(hook_options, 1):
    print(f"{i}. [{h['style']}] {h['hook']}")

In [ ]:
# ✏️  Change the index to pick a different hook
HOOK_INDEX = 0

chosen_hook = hook_options[HOOK_INDEX]
print(f"Chosen hook: {chosen_hook['hook']}")

## 7. Generate slides

In [ ]:
from src.content.generator import generate_slide_content

slides = generate_slide_content(
    topic=chosen_topic["title"],
    angle=angle,
    hook=chosen_hook["hook"],
    slide_count=slide_count,
    tone=tone,
    audience=audience,
    style_notes=style_notes,
    research_facts=research_facts or None,
)

for i, s in enumerate(slides, 1):
    print(f"--- Slide {i} ---")
    print(f"  Title:  {s.get('title', '')}")
    print(f"  Body:   {s.get('body', '')}")
    print(f"  Footer: {s.get('footer', '')}")
    print()

## 8. Review & improve engagement

In [ ]:
from src.content.reviewer import review_and_improve

slides = review_and_improve(
    slides=slides,
    tone=tone,
    audience=audience,
    iterations=review_iterations,
    hook=chosen_hook["hook"],
)
print("Review complete.")

## 9. Fact-check & validate

In [ ]:
from src.content.generator import (
    layered_fact_check,
    fact_check_slides,
    validate_conclusion,
    check_narrative_coherence,
    add_value_pass,
    strip_claim_tags,
)

# Layered fact-check
if research_facts:
    for fc_round in range(1, 4):
        fc_result = layered_fact_check(
            slides, research_text, research_facts,
            chosen_topic["title"], angle,
        )
        slides = fc_result.get("corrected_slides", slides)
        has_issues = any(
            item.get("status") in ("flagged", "unverifiable")
            for item in fc_result.get("layer_a_report", []) + fc_result.get("layer_b_report", [])
        )
        print(f"Round {fc_round}: {'issues found' if has_issues else 'all clear'}")
        if not has_issues:
            break
else:
    fc_result = fact_check_slides(slides, chosen_topic["title"], angle)
    slides = fc_result.get("corrected_slides", slides)

# Conclusion validation
conclusion_result = validate_conclusion(slides, research_facts or [], chosen_topic["title"], angle)
slides = conclusion_result.get("corrected_slides", slides)
print(f"Conclusion logic valid: {conclusion_result.get('logic_valid', True)}")

# Narrative coherence
coherence_result = check_narrative_coherence(slides, chosen_topic["title"], angle, chosen_hook["hook"])
slides = coherence_result.get("corrected_slides", slides)
print(f"Coherence score: {coherence_result.get('coherence_score', '?')}/10")

# Final value pass
value_result = add_value_pass(
    slides=slides, topic=chosen_topic["title"],
    angle=angle, audience=audience, hook=chosen_hook["hook"],
)
if isinstance(value_result, list):
    value_result = {"corrected_slides": value_result}
slides = value_result.get("corrected_slides", slides)

# Strip internal claim tags
slides = strip_claim_tags(slides)

## 10. Generate metadata & build PPTX

In [ ]:
from src.content.generator import generate_tiktok_metadata
from src.slides.pptx_builder import build_pptx

metadata = generate_tiktok_metadata(
    slides=slides,
    topic=chosen_topic["title"],
    angle=angle,
    hook=chosen_hook["hook"],
)
print(f"Title: {metadata.get('title', '')}")
print(f"Description: {metadata.get('description', '')}")

filepath = build_pptx(
    slides=slides,
    colors=colors,
    aspect_ratio=aspect_ratio,
    output_dir=output_dir,
)
print(f"\nSaved to: {filepath}")

## 11. Preview final slides

In [ ]:
for i, slide in enumerate(slides, 1):
    print(f"=== Slide {i} ===")
    print(f"  Title:  {slide.get('title', '')}")
    print(f"  Body:   {slide.get('body', '')}")
    print(f"  Footer: {slide.get('footer', '')}")
    print()